<a href="https://colab.research.google.com/github/Thishna/Thishna-Data_Science_Projects/blob/Data-Science/Thishna_Moving_object_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python opencv-python-headless

In [ ]:
!wget https://github.com/pjreddie/darknet/blob/master/cfg/yolov4.cfg?raw=true -O yolov4.cfg
!wget https://pjreddie.com/media/files/yolov4.weights
!wget https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names

--2025-01-01 16:45:09--  https://github.com/pjreddie/darknet/blob/master/cfg/yolov4.cfg?raw=true
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-01-01 16:45:09 ERROR 404: Not Found.

--2025-01-01 16:45:09--  https://pjreddie.com/media/files/yolov4.weights
Resolving pjreddie.com (pjreddie.com)... 162.0.215.52
Connecting to pjreddie.com (pjreddie.com)|162.0.215.52|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-01-01 16:45:10 ERROR 404: Not Found.

--2025-01-01 16:45:10--  https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 625 [text

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

!wget https://pjreddie.com/media/files/yolov4.weights
!wget https://github.com/pjreddie/darknet/blob/master/cfg/yolov4.cfg?raw=true -O yolov4.cfg
!wget https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names

--2025-01-01 16:45:10--  https://pjreddie.com/media/files/yolov4.weights
Resolving pjreddie.com (pjreddie.com)... 162.0.215.52
Connecting to pjreddie.com (pjreddie.com)|162.0.215.52|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-01-01 16:45:10 ERROR 404: Not Found.

--2025-01-01 16:45:11--  https://github.com/pjreddie/darknet/blob/master/cfg/yolov4.cfg?raw=true
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-01-01 16:45:11 ERROR 404: Not Found.

--2025-01-01 16:45:11--  https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 625 [text

In [ ]:
import os
print(os.listdir())

['.config', 'coco.names.3', 'yolov4.cfg', 'yolov3.cfg', 'yolov3.weights', 'labeled_output.mp4', 'coco.names.1', 'coco.names', 'coco.names.2', 'moving object detection system (1).mp4', 'moving object detection system.mp4', 'sample_data']


In [ ]:
!wget https://pjreddie.com/media/files/yolov3.weights
!wget https://github.com/pjreddie/darknet/blob/master/cfg/yolov3.cfg?raw=true -O yolov3.cfg
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")

--2025-01-01 16:45:11--  https://pjreddie.com/media/files/yolov3.weights
Resolving pjreddie.com (pjreddie.com)... 162.0.215.52
Connecting to pjreddie.com (pjreddie.com)|162.0.215.52|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248007048 (237M) [application/octet-stream]
Saving to: ‘yolov3.weights.1’

yolov3.weights.1    100%[===================>] 236.52M  32.8MB/s    in 7.7s    

2025-01-01 16:45:19 (30.5 MB/s) - ‘yolov3.weights.1’ saved [248007048/248007048]

--2025-01-01 16:45:19--  https://github.com/pjreddie/darknet/blob/master/cfg/yolov3.cfg?raw=true
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/pjreddie/darknet/raw/refs/heads/master/cfg/yolov3.cfg [following]
--2025-01-01 16:45:19--  https://github.com/pjreddie/darknet/raw/refs/heads/master/cfg/yolov3.cfg
Reusing existing connection to github.com:443.

In [ ]:
# Load class names
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

print(classes[:10])  # Display the first 10 classes for confirmation

['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light']


In [ ]:
def detect_and_label_objects(video_path, output_path="labeled_output.mp4"):
    # Open the video file
    cap = cv2.VideoCapture(video_path)

    # Define codec and create VideoWriter object for MP4
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # 'mp4v' codec for MP4 format
    out = cv2.VideoWriter(output_path, fourcc, 30.0, (640, 360))  # You can adjust the frame size

    while True:
        ret, frame = cap.read()
        if not ret:
            break  # Exit if no frame is read (end of video)

        # Resize the frame for better processing speed (optional)
        frame = cv2.resize(frame, (640, 360))

        # Prepare the image for YOLO
        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), (0, 0, 0), True, crop=False)
        net.setInput(blob)

        # Get the detections
        detections = net.forward(net.getUnconnectedOutLayersNames())

        # Loop through detections
        for detection in detections:
            for obj in detection:
                scores = obj[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]

                # Filter detections based on confidence threshold
                if confidence > 0.5:
                    label = str(classes[class_id])
                    x, y, w, h = map(int, obj[:4] * [640, 360, 640, 360])  # Rescale coordinates to frame size
                    cv2.rectangle(frame, (x, y), (x+100, y+100), (0, 255, 0), 2)  # Draw bounding box
                    cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)  # Label the object

        # Write the frame with labels to the output video
        out.write(frame)

    # Release resources
    cap.release()
    out.release()
    print(f"Labeled video saved as {output_path}")


In [ ]:
from google.colab import files
uploaded = files.upload()  # This will allow you to upload a video
video_path = list(uploaded.keys())[0]  # Get the uploaded video file name
print("Uploaded video:", video_path)

Saving moving object detection system.mp4 to moving object detection system (2).mp4
Uploaded video: moving object detection system (2).mp4


In [ ]:
detect_and_label_objects(video_path, "labeled_output.mp4")
from google.colab import files
files.download("labeled_output.mp4")

Labeled video saved as labeled_output.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>